In [2]:
## 2026 BARCELONA-CATALUNYA GRAND PRIX — PODIUM PREDICTION
### Rolling data: Australia (R1) + China (R2) + Japan (R3) + Miami (R4) + Canada (R5) + Monaco (R6)
### Barcelona GP qualifying grid sourced live (June 13 2026)
### Weather: HOT, ~33°C race day, DRY — high tyre degradation expected
###
### BARCELONA-SPECIFIC FACTORS ADDED vs Monaco model:
###   1. FP2 Long-Run Race Pace (medium tyre, 9-16 laps)
###   2. Tyre degradation index (turns 3 & 9 wear proxy, per team/car concept)
###   3. Pit stop performance score (reliability of stops under pressure)
###   4. Overtaking potential index (Barcelona allows more passing than Monaco)
###   5. Upgrade package score (Ferrari +8 new parts; penalises development risk)
###   6. Adjusted grid weight (0.20 vs Monaco's 0.25 — overtaking more possible)

import sys
!{sys.executable} -m pip install fastf1 xgboost --quiet

import fastf1
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

# ── Barcelona GP Qualifying Grid (June 13 2026) ───────────────────────────────
# Source: total-motorsport.com, gpfans.com, crash.net, racingnews365.com
# RUS pole: 1:14.679 | HAM +0.064 | ANT +0.319 | NOR +0.322 | VER +0.342
# HAD +0.398 | PIA +0.411 | LAW +1.863 | HUL +1.978 | LEC: no Q3 time (P10, crash)
# P11-P16: LIN, BOR, COL, GAS, BEA, SAI (Q2 elim)
# P17-P22: OCO, ALB, PER, BOT, STR, ALO (Q1 elim)
# NOTE: Leclerc crashed at T4 in Q3 → starts P10 with no representative Q3 time
# NOTE: Antonelli had battery issue in qualifying → may not reflect true pace

barcelona_quali_data = [
    {"Driver": "RUS", "BestLap_s": 74.679, "BarcelonaGrid": 1},
    {"Driver": "HAM", "BestLap_s": 74.743, "BarcelonaGrid": 2},
    {"Driver": "ANT", "BestLap_s": 74.998, "BarcelonaGrid": 3},
    {"Driver": "NOR", "BestLap_s": 75.001, "BarcelonaGrid": 4},
    {"Driver": "VER", "BestLap_s": 75.021, "BarcelonaGrid": 5},
    {"Driver": "HAD", "BestLap_s": 75.077, "BarcelonaGrid": 6},
    {"Driver": "PIA", "BestLap_s": 75.090, "BarcelonaGrid": 7},
    {"Driver": "LAW", "BestLap_s": 76.542, "BarcelonaGrid": 8},
    {"Driver": "HUL", "BestLap_s": 76.657, "BarcelonaGrid": 9},
    {"Driver": "LEC", "BestLap_s": 76.900, "BarcelonaGrid": 10},   # No Q3 time — estimated from Q2 pace + typical gap
    {"Driver": "LIN", "BestLap_s": 77.100, "BarcelonaGrid": 11},
    {"Driver": "BOR", "BestLap_s": 77.200, "BarcelonaGrid": 12},
    {"Driver": "COL", "BestLap_s": 77.350, "BarcelonaGrid": 13},
    {"Driver": "GAS", "BestLap_s": 77.500, "BarcelonaGrid": 14},
    {"Driver": "BEA", "BestLap_s": 77.650, "BarcelonaGrid": 15},
    {"Driver": "SAI", "BestLap_s": 77.800, "BarcelonaGrid": 16},
    {"Driver": "OCO", "BestLap_s": 78.100, "BarcelonaGrid": 17},
    {"Driver": "ALB", "BestLap_s": 78.250, "BarcelonaGrid": 18},
    {"Driver": "PER", "BestLap_s": 78.500, "BarcelonaGrid": 19},
    {"Driver": "BOT", "BestLap_s": 78.700, "BarcelonaGrid": 20},
    {"Driver": "STR", "BestLap_s": 78.900, "BarcelonaGrid": 21},
    {"Driver": "ALO", "BestLap_s": 79.100, "BarcelonaGrid": 22},
]

quali_df = pd.DataFrame(barcelona_quali_data)
pole_time = quali_df["BestLap_s"].min()
quali_df["BarcelonaGapFromPole_s"]   = (quali_df["BestLap_s"] - pole_time).round(3)
quali_df["BarcelonaGapFromPolePct"] = (quali_df["BarcelonaGapFromPole_s"] / pole_time * 100).round(4)

# Race-day weather
# Source: formula1.com, total-motorsport.com — extreme heat, 33°C air, ~52°C tarmac
rain_probability = 0.02   # effectively dry
temperature      = 33.0   # HOT — drives tyre deg significantly
quali_df["RainProbability"] = rain_probability
quali_df["Temperature"]     = temperature

print(quali_df[["Driver", "BestLap_s", "BarcelonaGrid",
                "BarcelonaGapFromPole_s", "BarcelonaGapFromPolePct"]])
print(f"\nConditions: {temperature}°C | Rain: {rain_probability:.0%} | EXTREME TYRE DEG expected")


# ── Driver Standings after Monaco GP (R6) — revised after Gasly penalty reinstatement ──
# Source: racefans.net, gpfans.com (updated June 12 2026)
driver_points = {
    "ANT": 156, "HAM": 90,  "RUS": 88,  "LEC": 75,
    "PIA": 58,  "NOR": 58,  "VER": 43,  "GAS": 35,
    "HAD": 26,  "LAW": 24,  "BEA": 18,  "COL": 15,
    "LIN": 11,  "SAI": 6,   "ALB": 5,   "OCO": 3,
    "BOR": 2,   "ALO": 1,   "HUL": 0,   "BOT": 0,
    "PER": 0,   "STR": 0,
}

# Constructor standings after Monaco GP (R6) — revised
# Source: racefans.net (June 12 2026)
team_points = {
    "Mercedes":      244,
    "Ferrari":       165,
    "McLaren":       116,
    "Red Bull":       69,
    "Alpine":         50,
    "Racing Bulls":   35,
    "Haas":           21,
    "Williams":       11,
    "Audi":            2,
    "Aston Martin":    1,
    "Cadillac":        0,
}

driver_to_team = {
    "RUS": "Mercedes",     "ANT": "Mercedes",
    "HAM": "Ferrari",      "LEC": "Ferrari",
    "NOR": "McLaren",      "PIA": "McLaren",
    "VER": "Red Bull",     "HAD": "Red Bull",
    "BEA": "Haas",         "OCO": "Haas",
    "LAW": "Racing Bulls", "LIN": "Racing Bulls",
    "HUL": "Audi",         "BOR": "Audi",
    "GAS": "Alpine",       "COL": "Alpine",
    "SAI": "Williams",     "ALB": "Williams",
    "PER": "Cadillac",     "BOT": "Cadillac",
    "ALO": "Aston Martin", "STR": "Aston Martin",
}

max_driver_pts = max(v for v in driver_points.values() if v > 0)
quali_df["DriverPoints"]     = quali_df["Driver"].map(driver_points).fillna(0)
quali_df["DriverPointsNorm"] = (quali_df["DriverPoints"] / max_driver_pts).round(4)


# ── Rolling Race Results: R1-R6 ───────────────────────────────────────────────
# All 6 races: AUS, CHN, JPN, MIA, CAN, MON

aus_finish = {
    "RUS": 1,  "ANT": 2,  "LEC": 3,  "HAM": 4,
    "NOR": 5,  "VER": 6,  "BEA": 7,  "LIN": 8,
    "BOR": 9,  "GAS": 10, "OCO": 11, "ALB": 12,
    "LAW": 13, "COL": 14, "SAI": 15, "PER": 16,
    "STR": 17, "ALO": 18, "BOT": 19, "HAD": 20,
    "PIA": 21, "HUL": 22,
}

china_finish = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,
    "BEA": 5,  "GAS": 6,  "LAW": 7,  "HAD": 8,
    "SAI": 9,  "COL": 10, "HUL": 11, "LIN": 12,
    "BOT": 13, "OCO": 14, "PER": 15, "VER": 16,
    "ALO": 17, "STR": 18, "NOR": 19, "BOR": 20,
    "ALB": 21, "PIA": 22,
}

japan_finish = {
    "ANT": 1,  "PIA": 2,  "LEC": 3,  "RUS": 4,
    "NOR": 5,  "HAM": 6,  "GAS": 7,  "VER": 8,
    "LAW": 9,  "OCO": 10, "HUL": 11, "HAD": 12,
    "BOR": 13, "LIN": 14, "SAI": 15, "COL": 16,
    "PER": 17, "ALO": 18, "BOT": 19, "ALB": 20,
    "STR": 21, "BEA": 22,
}

miami_finish = {
    "ANT": 1,  "NOR": 2,  "PIA": 3,  "RUS": 4,
    "VER": 5,  "HAM": 6,  "COL": 7,  "LEC": 8,
    "SAI": 9,  "ALB": 10, "BEA": 11, "BOR": 12,
    "OCO": 13, "LIN": 14, "ALO": 15, "PER": 16,
    "STR": 17, "BOT": 18, "HUL": 19, "LAW": 20,
    "GAS": 21, "HAD": 22,
}

canada_finish = {
    "ANT": 1,  "HAM": 2,  "VER": 3,  "LEC": 4,
    "HAD": 5,  "COL": 6,  "LAW": 7,  "GAS": 8,
    "SAI": 9,  "BEA": 10, "PIA": 11, "HUL": 12,
    "BOR": 13, "OCO": 14, "STR": 15, "BOT": 16,
    "ALB": 17, "ALO": 18, "LIN": 19, "NOR": 20,
    "RUS": 21, "PER": 22,
}

# Monaco GP result (revised — Gasly reinstated to P3)
# Source: racefans.net June 12 2026
# DNFs: LEC (brakes, P16+), VER (power off line P1), NOR (battery P17+), BOT (brakes)
monaco_finish = {
    "ANT": 1,  "HAM": 2,  "GAS": 3,  "HAD": 4,
    "PIA": 5,  "LAW": 6,  "LIN": 7,  "ALB": 8,
    "OCO": 9,  "ALO": 10, "BOR": 11, "RUS": 12,
    "HUL": 13, "COL": 14, "PER": 15,
    "VER": 17, "LEC": 18, "NOR": 19, "BOT": 20,
    "BEA": 21, "SAI": 22, "STR": 22,  # STR DNF / late DNS
}

# Qualifying grids for all 6 races
aus_grid = {
    "RUS": 1,  "ANT": 2,  "HAD": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "HAM": 7,  "LAW": 8,  "LIN": 9,  "BOR": 10,
    "OCO": 11, "HUL": 12, "ALB": 13, "GAS": 14, "COL": 15,
    "BEA": 16, "ALO": 17, "PER": 18, "BOT": 19, "VER": 20,
    "SAI": 21, "STR": 22,
}

china_grid = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "GAS": 7,  "VER": 8,  "HAD": 9,  "BEA": 10,
    "HUL": 11, "COL": 12, "OCO": 13, "LAW": 14, "LIN": 15,
    "BOR": 16, "SAI": 17, "ALB": 18, "ALO": 19, "BOT": 20,
    "STR": 21, "PER": 22,
}

japan_grid = {
    "ANT": 1,  "RUS": 2,  "PIA": 3,  "LEC": 4,  "NOR": 5,
    "HAM": 6,  "GAS": 7,  "HAD": 8,  "BOR": 9,  "LIN": 10,
    "VER": 11, "OCO": 12, "HUL": 13, "LAW": 14, "COL": 15,
    "SAI": 16, "ALB": 17, "BEA": 18, "PER": 19, "BOT": 20,
    "ALO": 21, "STR": 22,
}

miami_grid = {
    "ANT": 1,  "NOR": 2,  "VER": 3,  "LEC": 4,  "PIA": 5,
    "RUS": 6,  "GAS": 7,  "BEA": 8,  "SAI": 9,  "OCO": 10,
    "HUL": 11, "LAW": 12, "LIN": 13, "ALB": 14, "BOR": 15,
    "COL": 16, "ALO": 17, "STR": 18, "BOT": 19, "HAM": 20,
    "HAD": 21, "PER": 22,
}

canada_grid = {
    "RUS": 1,  "ANT": 2,  "NOR": 3,  "PIA": 4,  "HAM": 5,
    "VER": 6,  "HAD": 7,  "LEC": 8,  "LIN": 9,  "COL": 10,
    "HUL": 11, "LAW": 12, "BOR": 13, "GAS": 14, "SAI": 15,
    "BEA": 16, "OCO": 17, "ALB": 18, "ALO": 19, "PER": 20,
    "BOT": 21, "STR": 22,
}

# Monaco grid (qualifying)
monaco_grid = {
    "ANT": 1,  "VER": 2,  "HAM": 3,  "LEC": 4,  "HAD": 5,
    "RUS": 6,  "PIA": 7,  "NOR": 8,  "GAS": 9,  "LAW": 10,
    "ALB": 11, "SAI": 12, "HUL": 13, "COL": 14, "LIN": 15,
    "BOR": 16, "OCO": 17, "PER": 18, "BEA": 19, "BOT": 20,
    "ALO": 21, "STR": 22,
}

# Barcelona qualifying grid
barcelona_grid = {
    "RUS": 1,  "HAM": 2,  "ANT": 3,  "NOR": 4,  "VER": 5,
    "HAD": 6,  "PIA": 7,  "LAW": 8,  "HUL": 9,  "LEC": 10,
    "LIN": 11, "BOR": 12, "COL": 13, "GAS": 14, "BEA": 15,
    "SAI": 16, "OCO": 17, "ALB": 18, "PER": 19, "BOT": 20,
    "STR": 21, "ALO": 22,
}

# ── Build 6-race rolling features ─────────────────────────────────────────────
drivers = list(driver_to_team.keys())
N = len(drivers)  # 22

records = []
for drv in drivers:
    aus_fp  = aus_finish.get(drv, 20)
    chn_fp  = china_finish.get(drv, 20)
    jpn_fp  = japan_finish.get(drv, 20)
    mia_fp  = miami_finish.get(drv, 20)
    can_fp  = canada_finish.get(drv, 20)
    mon_fp  = monaco_finish.get(drv, 20)

    aus_gp  = aus_grid.get(drv, 20)
    chn_gp  = china_grid.get(drv, 20)
    jpn_gp  = japan_grid.get(drv, 20)
    mia_gp  = miami_grid.get(drv, 20)
    can_gp  = canada_grid.get(drv, 20)
    mon_gp  = monaco_grid.get(drv, 20)
    bcn_gp  = barcelona_grid.get(drv, 20)

    avg_finish_norm = round(
        ((aus_fp-1) + (chn_fp-1) + (jpn_fp-1) +
         (mia_fp-1) + (can_fp-1) + (mon_fp-1)) / (6 * (N-1)), 4
    )

    avg_grid_norm = round(
        ((aus_gp-1) + (chn_gp-1) + (jpn_gp-1) +
         (mia_gp-1) + (can_gp-1) + (mon_gp-1)) / (6 * (N-1)), 4
    )

    # Exponential decay recent form: BCN Q 30%, MON 25%, CAN 18%, MIA 12%, JPN 8%, CHN 5%, AUS 2%
    recent_finish_norm = round(
        0.30*(bcn_gp-1)/(N-1) +   # Barcelona qualifying as most recent signal
        0.25*(mon_fp-1)/(N-1) +
        0.18*(can_fp-1)/(N-1) +
        0.12*(mia_fp-1)/(N-1) +
        0.08*(jpn_fp-1)/(N-1) +
        0.05*(chn_fp-1)/(N-1) +
        0.02*(aus_fp-1)/(N-1), 4
    )

    # DNF rate (6 races)
    dnf_flags = [
        1 if aus_fp >= 18 else 0,
        1 if chn_fp >= 16 else 0,
        1 if jpn_fp >= 21 else 0,
        1 if mia_fp >= 19 else 0,
        1 if can_fp >= 20 else 0,
        1 if mon_fp >= 16 else 0,
    ]
    dnf_rate = round(sum(dnf_flags) / 6, 3)

    records.append({
        "Driver":           drv,
        "Team":             driver_to_team[drv],
        "AusFinish":        aus_fp,  "ChinaFinish": chn_fp,
        "JapanFinish":      jpn_fp,  "MiamiFinish": mia_fp,
        "CanadaFinish":     can_fp,  "MonacoFinish": mon_fp,
        "AvgFinishNorm":    avg_finish_norm,
        "RecentFinishNorm": recent_finish_norm,
        "AvgGridNorm":      avg_grid_norm,
        "BarcelonaGrid":    bcn_gp,
        "DNF_rate":         dnf_rate,
    })

rolling_df = pd.DataFrame(records)

max_team_pts   = max(team_points.values())
max_driver_pts = max(driver_points.values())

rolling_df["TeamScore"] = rolling_df["Team"].map(
    {t: max(p, 0.5) / max_team_pts for t, p in team_points.items()}
).round(4)

rolling_df["DriverPointsNorm"] = rolling_df["Driver"].map(
    {d: p / max_driver_pts for d, p in driver_points.items()}
).round(4)

print("\n" + "=" * 120)
print("  ROLLING FEATURES FOR BARCELONA GP PREDICTION")
print("  Based on: Australia (R1) + China (R2) + Japan (R3) + Miami (R4) + Canada (R5) + Monaco (R6)")
print("=" * 120)


# ── BARCELONA-SPECIFIC FEATURE 1: FP2 Long-Run Race Pace ─────────────────────
# Source: total-motorsport.com, autosport.com (FP2 medium-tyre long runs)
# RUS: 1:21.571 avg / 10 laps (medium) → fastest; LEC: 1:21.697 / 9 laps (+0.126)
# VER: ~1:21.818 / ~8 laps (soft, adjust ≈ +0.5 for compound delta → ~1:22.318)
# McLaren: ~0.458 slower than RUS on softs → ~1:22.0 equiv on mediums
# Audi (HUL): 1:22.857 / 10 laps (medium) — class of midfield
# GAS (Alpine): 1:23.636 / 14 laps (medium)
# RB (LIN): 1:23.826 / 16 laps (medium)
# Williams (ALB): 1:24.552 / 10 laps (soft → adjust +0.3 = ~1:24.852 on mediums)
# Cadillac (PER): 1:24.724 / 11 laps (medium)
# HAM had limited FP2 data (Ferrari ran different programme, explained by HAM "engine mode")
# ANT sat out FP1, battery issue in qualifying

fp2_long_run_data = [
    # Driver, avg medium-equivalent gap to RUS (0 = fastest)
    # Top teams
    {"Driver": "RUS", "FP2_LongRun_GapToFastest_s": 0.000},   # 1:21.571 — benchmark
    {"Driver": "LEC", "FP2_LongRun_GapToFastest_s": 0.126},   # 1:21.697 — Ferrari race pace strong
    {"Driver": "HAM", "FP2_LongRun_GapToFastest_s": 0.200},   # estimated; limited FP2 data, different programme
    {"Driver": "ANT", "FP2_LongRun_GapToFastest_s": 0.180},   # Mercedes race pace historically dominant; +0.18 estimate
    {"Driver": "VER", "FP2_LongRun_GapToFastest_s": 0.247},   # 0.247s on soft; soft-to-med delta applied
    {"Driver": "HAD", "FP2_LongRun_GapToFastest_s": 0.280},   # Red Bull pair, estimated
    # McLaren — race pace weakest of top teams
    {"Driver": "NOR", "FP2_LongRun_GapToFastest_s": 0.458},   # 0.458s behind on long runs (soft adjusted)
    {"Driver": "PIA", "FP2_LongRun_GapToFastest_s": 0.480},
    # Midfield leaders
    {"Driver": "HUL", "FP2_LongRun_GapToFastest_s": 1.286},   # 1:22.857 Audi — best midfield
    {"Driver": "BOR", "FP2_LongRun_GapToFastest_s": 1.400},   # Audi pair
    {"Driver": "GAS", "FP2_LongRun_GapToFastest_s": 2.065},   # Alpine 1:23.636
    {"Driver": "COL", "FP2_LongRun_GapToFastest_s": 2.200},   # Alpine pair (fewer laps)
    {"Driver": "LAW", "FP2_LongRun_GapToFastest_s": 2.255},   # Racing Bulls 1:23.826
    {"Driver": "LIN", "FP2_LongRun_GapToFastest_s": 2.300},
    {"Driver": "BEA", "FP2_LongRun_GapToFastest_s": 2.319},   # Haas ≈ Alpine + 0.054
    {"Driver": "OCO", "FP2_LongRun_GapToFastest_s": 2.380},
    # Williams, Cadillac, Aston Martin
    {"Driver": "SAI", "FP2_LongRun_GapToFastest_s": 3.100},   # Williams 1:24.552 soft → ~1:24.85 med adj
    {"Driver": "ALB", "FP2_LongRun_GapToFastest_s": 3.200},
    {"Driver": "PER", "FP2_LongRun_GapToFastest_s": 3.153},   # Cadillac 1:24.724
    {"Driver": "BOT", "FP2_LongRun_GapToFastest_s": 3.250},
    {"Driver": "ALO", "FP2_LongRun_GapToFastest_s": 3.800},   # Aston Martin — limited data
    {"Driver": "STR", "FP2_LongRun_GapToFastest_s": 3.900},
]

fp2_df = pd.DataFrame(fp2_long_run_data)
print("\nFP2 Long-Run Pace (medium-equivalent gap to Russell):")
print(fp2_df.sort_values("FP2_LongRun_GapToFastest_s").to_string(index=False))


# ── BARCELONA-SPECIFIC FEATURE 2: Tyre Degradation Index ─────────────────────
# Barcelona Turns 3 (heavy lateral load → rear deg) and Turn 9 (long medium-speed
# corner → front-left deg) are the primary wear generators.
# Scale 0 (lowest deg) → 1 (highest deg).
# Based on 2024-2025 historical deg rates + 2026 car/concept characteristics.
# New 2026 regs favour cars with low floor ride height sensitivity (Mercedes advantage).
# High downforce setups (Ferrari upgrade package) may worsen deg under extreme heat.
# McLaren historically poor deg at Barcelona; Red Bull better on this metric.
#
# Sources: historical Pirelli tyre data, 2025 Barcelona race, FP2 tyre observations,
# pirelli.com compound selection notes (mediums + hards for 2026)

tyre_deg_index = {
    # Low degradation = better for Barcelona
    "Mercedes":      0.15,   # best tyre management on new regs; cool platform
    "Red Bull":      0.22,   # strong deg historically at Barcelona
    "Ferrari":       0.30,   # 8 new parts risk = higher initial deg uncertainty; strong base package
    "Racing Bulls":  0.38,   # RB junior team, similar concept
    "Haas":          0.40,
    "Audi":          0.42,
    "Alpine":        0.45,
    "McLaren":       0.48,   # historically worst deg at Barcelona despite overall speed
    "Williams":      0.52,
    "Cadillac":      0.55,
    "Aston Martin":  0.60,
}

rolling_df["TyreDegIndex"] = rolling_df["Team"].map(tyre_deg_index).round(3)


# ── BARCELONA-SPECIFIC FEATURE 3: Pit Stop Performance Score ─────────────────
# Based on 2026 pit stop times across R1-R6.
# Formula: lower score = faster/more reliable stops.
# Sources: 2026 pitstop data, bestf1.com, formula1.com
# Monaco penalty chaos context: Hamilton, Russell, Piastri, Gasly penalised for pit
# entry speeding → team procedure reliability matters.

pitstop_score = {
    # (avg stop time in seconds + reliability penalty for errors)
    # Lower = better
    "Mercedes":      2.35,   # fast stops historically; Monaco penalties for pit lane speeding (procedural not mechanical)
    "Ferrari":       2.45,   # solid stops in 2026
    "McLaren":       2.40,   # strong but Monaco penalised (pit entry)
    "Red Bull":      2.50,
    "Alpine":        2.65,   # Gasly Monaco pit lane penalty (procedural)
    "Racing Bulls":  2.58,
    "Haas":          2.70,
    "Audi":          2.72,
    "Williams":      2.80,
    "Cadillac":      2.90,
    "Aston Martin":  2.95,
}

rolling_df["PitStopScore"] = rolling_df["Team"].map(pitstop_score).round(3)


# ── BARCELONA-SPECIFIC FEATURE 4: Overtaking Potential Index ─────────────────
# Barcelona allows more overtaking than Monaco. Key zones: T1, T4-5 complex,
# Long straight to T10 (DRS zone). Track position less absolute but still important.
# Scale: 0 = low overtaking ability, 1 = high overtaking ability
# Based on: historical position changes per race at Barcelona + 2026 straight-line
# speed estimates (active aero / biturbo hybrid system = new regs favour racers)

overtake_index = {
    # Higher = better ability to overtake / make positions
    "ANT": 0.88,   # consistently makes positions, strong race craft
    "HAM": 0.92,   # best overtaker on grid, 7x WC
    "RUS": 0.78,
    "LEC": 0.75,
    "NOR": 0.80,
    "PIA": 0.76,
    "VER": 0.85,
    "HAD": 0.72,
    "LAW": 0.68,
    "HUL": 0.65,
    "GAS": 0.63,
    "COL": 0.60,
    "LIN": 0.58,
    "BEA": 0.55,
    "SAI": 0.70,
    "ALB": 0.62,
    "BOR": 0.55,
    "OCO": 0.60,
    "PER": 0.58,
    "BOT": 0.50,
    "STR": 0.45,
    "ALO": 0.72,   # experience / racecraft
}

rolling_df["OvertakeIndex"] = rolling_df["Driver"].map(overtake_index).fillna(0.55).round(3)


# ── BARCELONA-SPECIFIC FEATURE 5: Upgrade Package Score ─────────────────────
# Ferrari brought 8 new parts to Barcelona.
# Risk: first race with combined package = limited long-run data, potential balance issues.
# Reward: if successful, closes pace gap further.
# Score: positive = net benefit expected; negative = risk outweighs short-term gain.
# Source: total-motorsport.com, formula1.com (Ferrari upgrade narrative)

upgrade_score = {
    # Net upgrade impact this race (normalised -1 to 1; positive = benefit expected)
    "Mercedes":      0.00,   # no major upgrade; car already dominant
    "Ferrari":       0.25,   # 8 new parts = potential gain; uncertainty adds risk (net positive)
    "McLaren":       0.05,   # incremental
    "Red Bull":      0.10,   # steady updates
    "Alpine":        0.00,
    "Racing Bulls":  0.05,
    "Haas":          0.00,
    "Audi":          0.08,   # Audi strong relative pace improvement in 2026
    "Williams":      0.00,
    "Cadillac":      0.00,
    "Aston Martin": -0.05,   # struggling; no sign of improvement
}

rolling_df["UpgradeScore"] = rolling_df["Team"].map(upgrade_score).fillna(0).round(3)


# ── BARCELONA-SPECIFIC FEATURE 6: Heat Management Index ──────────────────────
# 33°C air / ~52°C tarmac predicted. Extreme heat amplifies tyre deg and
# puts pressure on energy management (active aero / hybrid battery cooling).
# Cars with superior energy management will benefit in the second half of the race.
# Scale: 0 (poor heat management) → 1 (excellent heat management)

heat_mgmt_index = {
    "Mercedes":      0.90,   # historically excellent energy/temp management
    "Ferrari":       0.75,
    "Red Bull":      0.78,
    "McLaren":       0.65,   # weak in hot races historically
    "Alpine":        0.68,
    "Racing Bulls":  0.70,
    "Haas":          0.62,
    "Audi":          0.66,
    "Williams":      0.58,
    "Cadillac":      0.55,
    "Aston Martin":  0.52,
}

rolling_df["HeatMgmtIndex"] = rolling_df["Team"].map(heat_mgmt_index).round(3)


# ── Merge all data ────────────────────────────────────────────────────────────
df = quali_df.merge(
    rolling_df[[
        "Driver", "Team", "AvgFinishNorm", "RecentFinishNorm", "AvgGridNorm",
        "DNF_rate", "TeamScore", "TyreDegIndex", "PitStopScore",
        "OvertakeIndex", "UpgradeScore", "HeatMgmtIndex"
    ]],
    on="Driver", how="left"
)

df_final = df.merge(
    fp2_df[["Driver", "FP2_LongRun_GapToFastest_s"]],
    on="Driver", how="left"
)


# ── Feature Engineering ───────────────────────────────────────────────────────
# Composite Barcelona-specific features

# Tyre management composite (lower = better race outcome)
# Combines: deg index + heat management (inverse) + pitstop score normalised
df_final["TyreMgmtComposite"] = (
    df_final["TyreDegIndex"] * 0.40 +
    (1 - df_final["HeatMgmtIndex"]) * 0.35 +
    (df_final["PitStopScore"] / df_final["PitStopScore"].max()) * 0.25
).round(4)

# Race pace composite (lower = faster)
# FP2 long run weighted with tyre deg for sustained pace
max_fp2 = df_final["FP2_LongRun_GapToFastest_s"].max()
df_final["RacePaceComposite"] = (
    (df_final["FP2_LongRun_GapToFastest_s"] / max_fp2) * 0.65 +
    df_final["TyreDegIndex"] * 0.35
).round(4)

# Qualifying advantage with upgrade potential
df_final["QualiUpgradeBonus"] = (
    (df_final["BarcelonaGapFromPolePct"] / df_final["BarcelonaGapFromPolePct"].max()) -
    df_final["UpgradeScore"] * 0.1
).round(4)

feature_cols = [
    # Qualifying performance
    "BarcelonaGapFromPole_s",    # Gap to pole in Barcelona quali
    "BarcelonaGrid",             # Grid position (moderate weight — overtaking possible)
    "QualiUpgradeBonus",         # Quali gap adjusted for upgrade potential

    # FP2 long-run race pace
    "FP2_LongRun_GapToFastest_s",  # KEY at Barcelona — 66 laps, race pace matters
    "RacePaceComposite",            # Race pace + tyre deg combined

    # Barcelona-specific tyre/heat factors
    "TyreDegIndex",              # Tyre deg susceptibility (Turns 3 & 9)
    "HeatMgmtIndex",             # 33°C heat management
    "TyreMgmtComposite",         # Combined tyre management score
    "PitStopScore",              # Pit stop speed/reliability
    "OvertakeIndex",             # Ability to make positions in race

    # Constructor & championship signals
    "TeamScore",                 # Constructor points after Monaco R6 (normalised)
    "DriverPointsNorm",          # Driver points after Monaco R6 (normalised)
    "UpgradeScore",              # Net upgrade benefit this race

    # Rolling form
    "AvgFinishNorm",             # Rolling avg finish (R1-R6)
    "RecentFinishNorm",          # Weighted recent form (BCN quali 30%, MON 25%...)
    "AvgGridNorm",               # Rolling avg grid (R1-R6)
    "DNF_rate",                  # Reliability (6 races)

    # Weather
    "RainProbability",           # ~0% rain
    "Temperature",               # 33°C — amplifies all deg/heat features
]


# ── Target Variable ───────────────────────────────────────────────────────────
# Barcelona: grid LESS dominant than Monaco (overtaking possible)
# Race pace and tyre management MORE important than Monaco
# Grid weight reduced to 0.20 (vs 0.25 Monaco); recent form stays high
df_final["RaceScore"] = (
    0.40 * df_final["AvgFinishNorm"] +
    0.30 * df_final["RecentFinishNorm"] +
    0.20 * df_final["AvgGridNorm"] +
    0.10 * df_final["TyreMgmtComposite"]   # Barcelona-specific addition
).round(4)

X = df_final[feature_cols].copy()
y = df_final["RaceScore"]

imputer   = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)


# ── Multi-model LOO-CV ────────────────────────────────────────────────────────
models = {
    "Ridge":         Ridge(alpha=1.0),
    "ElasticNet":    ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest":  RandomForestRegressor(
                         n_estimators=150, max_depth=4,
                         min_samples_leaf=3, random_state=42),
    "GradientBoost": GradientBoostingRegressor(
                         n_estimators=150, learning_rate=0.05,
                         max_depth=3, subsample=0.8, random_state=42),
    "XGBoost":       XGBRegressor(
                         n_estimators=150, learning_rate=0.05,
                         max_depth=3, subsample=0.8,
                         colsample_bytree=0.8, reg_lambda=2.0,
                         random_state=42),
}

loo     = LeaveOneOut()
results = {}

for name, mdl in models.items():
    pipe = (Pipeline([("scaler", StandardScaler()), ("model", mdl)])
            if name in ["Ridge", "ElasticNet"]
            else Pipeline([("model", mdl)]))
    errors = []
    for train_idx, test_idx in loo.split(X_imputed):
        pipe.fit(X_imputed[train_idx], y.iloc[train_idx])
        errors.append(abs(pipe.predict(X_imputed[test_idx])[0] - y.iloc[test_idx].iloc[0]))
    results[name] = {"MAE": round(np.mean(errors), 4), "StdDev": round(np.std(errors), 4)}

print(f"\n{'Model':<18} {'LOO MAE':>8} {'Std Dev':>8}  {'Verdict'}")
print("─" * 55)
for name, res in sorted(results.items(), key=lambda x: x[1]["MAE"]):
    best = " ← best" if res["MAE"] == min(r["MAE"] for r in results.values()) else ""
    print(f"{name:<18} {res['MAE']:>8.4f} {res['StdDev']:>8.4f}  {best}")


# ── Best model final prediction ───────────────────────────────────────────────
best_name  = min(results, key=lambda x: results[x]["MAE"])
best_model = models[best_name]
final_pipe = (Pipeline([("scaler", StandardScaler()), ("model", best_model)])
              if best_name in ["Ridge", "ElasticNet"]
              else Pipeline([("model", best_model)]))
final_pipe.fit(X_imputed, y)
df_final["PredictedScore"] = final_pipe.predict(X_imputed)

final = df_final.sort_values("PredictedScore").reset_index(drop=True)

print("\n" + "=" * 78)
print("   PREDICTED FINISHING ORDER — 2026 BARCELONA-CATALUNYA GRAND PRIX")
print("   Rolling: AUS + CHN + JPN + MIA + CAN + MON | + Barcelona-specific features")
print("=" * 78)
print(f"  {'P':<4}{'DRV':<6}{'Team':<22}{'BCN Grid':<10}{'Score'}")
print("  " + "─" * 55)
for i, row in final.iterrows():
    print(f"  {i+1:<4}{row['Driver']:<6}{row['Team']:<22}"
          f"P{row['BarcelonaGrid']:<9}{row['PredictedScore']:.4f}")

podium = final.head(3)
print(f"\n  {'='*58}")
print(f"  🥇 P1: {podium.iloc[0]['Driver']} ({podium.iloc[0]['Team']})")
print(f"  🥈 P2: {podium.iloc[1]['Driver']} ({podium.iloc[1]['Team']})")
print(f"  🥉 P3: {podium.iloc[2]['Driver']} ({podium.iloc[2]['Team']})")
print(f"\n  Best model:   {best_name}")
print(f"  LOO MAE:      {results[best_name]['MAE']:.4f}")
print(f"  Weather:      {df_final['Temperature'].iloc[0]:.0f}°C | "
      f"Rain: {df_final['RainProbability'].iloc[0]:.0%} | EXTREME HEAT")
print(f"  Grid weight:  0.20 (vs 0.25 Monaco — overtaking possible at BCN)")
print(f"  Key factors:  FP2 long-run race pace | Tyre deg T3+T9 | Heat mgmt")
print(f"  Ferrari note: 8 new upgrade parts — pace gain likely, some uncertainty")
print(f"  LEC note:     Starts P10 (Q3 crash) — needs Turn 1 fortune / strategy")
print(f"  ANT note:     Battery issue in quali but historically dominant on race pace")
print(f"  {'='*58}")


# ── Feature importances ───────────────────────────────────────────────────────
if best_name in ["RandomForest", "GradientBoost", "XGBoost"]:
    importances = final_pipe.named_steps["model"].feature_importances_
    print("\nFeature importances:")
    for feat, imp in sorted(zip(feature_cols, importances), key=lambda x: -x[1]):
        bar = "█" * int(imp * 50)
        print(f"  {feat:<36} {imp:.4f}  {bar}")
elif best_name in ["Ridge", "ElasticNet"]:
    coefficients = final_pipe.named_steps["model"].coef_
    print("\nFeature coefficients:")
    for feat, coef in sorted(zip(feature_cols, coefficients), key=lambda x: -abs(x[1])):
        print(f"  {feat:<36} {coef:+.4f}")


# ── All-models side-by-side ───────────────────────────────────────────────────
all_predictions = {}
for name, mdl in models.items():
    pipe = (Pipeline([("scaler", StandardScaler()), ("model", mdl)])
            if name in ["Ridge", "ElasticNet"]
            else Pipeline([("model", mdl)]))
    pipe.fit(X_imputed, y)
    ranked = pd.Series(pipe.predict(X_imputed),
                       index=df_final["Driver"]).rank(method="min").astype(int)
    all_predictions[name] = ranked

pred_df = pd.DataFrame(all_predictions)
pred_df["BestModelPos"] = pd.Series(
    final_pipe.predict(X_imputed), index=df_final["Driver"]
).rank(method="min").astype(int)
pred_df = pred_df.sort_values("BestModelPos")

print("\n" + "=" * 108)
print("   PREDICTED FINISHING ORDER — 2026 BARCELONA GP  (all models)")
print("=" * 108)
col_w  = 14
header = f"  {'DRV':<6}{'Team':<22}"
for name in models: header += f"{name:>{col_w}}"
header += f"  {'★ Best':>{col_w}}"
print(header)
print("  " + "─" * 100)
for driver in pred_df.index:
    team = df_final.loc[df_final["Driver"] == driver, "Team"].values[0]
    bcn_grid = df_final.loc[df_final["Driver"] == driver, "BarcelonaGrid"].values[0]
    row  = f"  {driver:<6}{team:<22}"
    for name in models:
        row += f"  P{pred_df.loc[driver, name]:<{col_w-2}}"
    row += f"  P{pred_df.loc[driver, 'BestModelPos']:<{col_w-2}}"
    print(row)

# ── Consensus podium ──────────────────────────────────────────────────────────
best_order = pred_df.sort_values("BestModelPos")
p1, p2, p3 = best_order.index[0], best_order.index[1], best_order.index[2]
t1 = df_final.loc[df_final["Driver"] == p1, "Team"].values[0]
t2 = df_final.loc[df_final["Driver"] == p2, "Team"].values[0]
t3 = df_final.loc[df_final["Driver"] == p3, "Team"].values[0]

print(f"\n  {'='*65}")
print(f"  Best model: {best_name} (LOO MAE: {results[best_name]['MAE']:.4f})")
print(f"  🥇 P1: {p1} ({t1})")
print(f"  🥈 P2: {p2} ({t2})")
print(f"  🥉 P3: {p3} ({t3})")
print(f"  Weather:    {df_final['Temperature'].iloc[0]:.0f}°C | Rain: {df_final['RainProbability'].iloc[0]:.0%}")

print("\n  CONSENSUS PODIUM (across all models):")
print("  " + "─" * 45)
medals = ["🥇", "🥈", "🥉"]
for pos in [1, 2, 3]:
    pos_counts = {}
    for name in models:
        for d in pred_df[pred_df[name] == pos].index:
            pos_counts[d] = pos_counts.get(d, 0) + 1
    if pos_counts:
        top   = max(pos_counts, key=pos_counts.get)
        votes = pos_counts[top]
        print(f"  {medals[pos-1]} P{pos}: {top} — {votes}/{len(models)} models agree")
print(f"  {'='*65}")

print("""
  ┌─────────────────────────────────────────────────────────────────┐
  │  BARCELONA-SPECIFIC FACTORS SUMMARY                             │
  ├─────────────────────────────────────────────────────────────────┤
  │  FP2 Long Run:   RUS best, LEC P2 (+0.126s), ANT ~+0.18        │
  │  Tyre Deg:       Mercedes lowest, McLaren highest risk          │
  │  Turns 3 & 9:   Rear/front-left wear — long-run tyre choice key│
  │  Heat (33°C):   Amplifies deg; 2-stop likely for most teams     │
  │  Pit Strategy:  Mercedes fastest stops; Alpine/Cadillac slower  │
  │  Upgrade risk:  Ferrari 8 new parts — race balance TBD          │
  │  LEC impact:    Starting P10, strong race pace → recovery likely │
  │  ANT factor:    Battery issue quali, but dominant race pace      │
  └─────────────────────────────────────────────────────────────────┘
""")


   Driver  BestLap_s  BarcelonaGrid  BarcelonaGapFromPole_s  \
0     RUS     74.679              1                   0.000   
1     HAM     74.743              2                   0.064   
2     ANT     74.998              3                   0.319   
3     NOR     75.001              4                   0.322   
4     VER     75.021              5                   0.342   
5     HAD     75.077              6                   0.398   
6     PIA     75.090              7                   0.411   
7     LAW     76.542              8                   1.863   
8     HUL     76.657              9                   1.978   
9     LEC     76.900             10                   2.221   
10    LIN     77.100             11                   2.421   
11    BOR     77.200             12                   2.521   
12    COL     77.350             13                   2.671   
13    GAS     77.500             14                   2.821   
14    BEA     77.650             15                   2